# 03 · Chronological splits

Define the v1 champion / v2 challenger windows from the build plan. Counts stay in MySQL. The final Aug 5–21 holdout is never used for HPO.

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt

from cross_model_drift.notebook import setup_eda
from cross_model_drift.splits import split_summary_rows

nb = setup_eda()
config, table, read_sql = nb.config, nb.table, nb.read_sql
plan = nb.split_plan()
pd.DataFrame(split_summary_rows(plan))

## Window counts

In [ ]:
def window_sql(alias: str, window) -> str:
    start = window.start.isoformat()
    end_exclusive = window.end_exclusive_ts.date().isoformat()
    return f"""
    SELECT
        '{alias}' AS split,
        COUNT(*) AS n_tx,
        SUM(anti_fraud_status = 'positive') AS n_fraud,
        SUM(anti_fraud_status = 'positive') / COUNT(*) AS fraud_rate,
        MIN(created) AS created_min,
        MAX(created) AS created_max
    FROM `{table}`
    WHERE created >= '{start}' AND created < '{end_exclusive}'
    """


parts = []
for version in ("v1", "v2"):
    mapping = plan.windows_for(version)
    for name, window in mapping.items():
        parts.append(window_sql(f"{version}/{name}", window))

counts = read_sql(" UNION ALL ".join(parts))
counts

## Holdout isolation

v2 test and the final comparison window must be the same Aug 5–21 slice.

In [ ]:
holdout = plan.holdout
assert plan.v2["test"].start == holdout.start
assert plan.v2["test"].end == holdout.end
assert plan.v1["holdout"].start == holdout.start

overlap = read_sql(
    f"""
    SELECT
        SUM(created >= '{plan.v1["train"].start}' AND created < '{plan.v1["train"].end_exclusive_ts.date()}') AS v1_train,
        SUM(created >= '{holdout.start}' AND created < '{holdout.end_exclusive_ts.date()}') AS holdout,
        SUM(
            created >= '{plan.v1["train"].start}' AND created < '{plan.v1["train"].end_exclusive_ts.date()}'
            AND created >= '{holdout.start}' AND created < '{holdout.end_exclusive_ts.date()}'
        ) AS overlap
    FROM `{table}`
    """
)
assert int(overlap.loc[0, "overlap"]) == 0
overlap

## Daily volume vs split boundaries

In [ ]:
daily = read_sql(
    f"""
    SELECT DATE(created) AS day, COUNT(*) AS n_tx
    FROM `{table}`
    GROUP BY DATE(created)
    ORDER BY day
    """
)
daily["day"] = pd.to_datetime(daily["day"])

fig, ax = plt.subplots(figsize=(12, 4.2))
ax.plot(daily["day"], daily["n_tx"], color="#4C78A8", linewidth=1.2)
for window, color, label in (
    (plan.v1["train"], "#4C78A8", "v1 train"),
    (plan.v1["validation"], "#72B7B2", "v1 valid"),
    (plan.v1["test"], "#F58518", "v1 test"),
    (plan.holdout, "#E45756", "final holdout"),
):
    ax.axvspan(window.start_ts, window.end_exclusive_ts, color=color, alpha=0.12, label=label)
ax.set_title("Daily volume with chronological windows")
ax.set_ylabel("transactions")
ax.legend(loc="upper left", ncol=2)
nb.show(fig)